In [ ]:
import os
import re
import pymupdf
import requests
from tqdm import tqdm
import ollama
import json
import io       # Для работы с байтами PDF из сети
import nltk     # Для токенизации предложений
import hashlib  # Для создания уникальных ID
import time
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cosine

## Load

In [ ]:
from langchain.document_loaders import TextLoader

PDF_URL = "https://www.bulgakov.ru/pdf/Master-i-Margarita.pdf"

In [ ]:
def fetch_and_extract_text_from_url(pdf_url):
    """
    Загружает PDF по URL и извлекает из него текст.
    """
    print(f"Fetching PDF from {pdf_url}...")
    try:
        response = requests.get(pdf_url, timeout=30)
        response.raise_for_status()

        pdf_bytes = io.BytesIO(response.content)

        print("Extracting text from PDF...")
        text = ""

        with pymupdf.open(stream=pdf_bytes, filetype="pdf") as document:
            for page_num in tqdm(range(len(document)), desc="Processing PDF pages"):
                page = document.load_page(page_num)
                text += page.get_text("text")

        text = re.sub(r'\s+', ' ', text).strip()
        print(f"Successfully extracted {len(text)} characters.")
        return text
    except requests.exceptions.RequestException as e:
        print(f"Error fetching PDF: {e}")
        return None
    except Exception as e:
        print(f"An error occurred during PDF processing: {e}")
        return None

In [ ]:
full_text = fetch_and_extract_text_from_url(PDF_URL)
if full_text:
    print("First 500 chars:", full_text[1500: 2000])

Fetching PDF from https://www.bulgakov.ru/pdf/Master-i-Margarita.pdf...
Extracting text from PDF...


Processing PDF pages: 100%|██████████| 451/451 [00:01<00:00, 253.41it/s]


Successfully extracted 759797 characters.
First 500 chars: ание и вечный приют / 433 Эпилог / 439 ЧАСТЬ ПЕРВАЯ  … Так кто ж ты, наконец? – Я – часть той силы, что вечно хочет зла и вечно совершает благо. Гете, «Фауст» Глава I Никогда не разговаривайте с неизвестными Однажды весною, в час небывало жаркого заката, в Москве, на Патриарших прудах, появились два гражданина. Первый из них, одетый в летнюю серенькую пару, был маленького рос- та, упитан, лыс, свою приличную шляпу пирожком нес в руке, а на хорошо выбритом лице его помещались сверхъестествен- ны


In [ ]:
try:
    client = ollama.Client()
    client.list()
    print(f"Ollama client initialized. Available models include those needed for script if pulled.")
except Exception as e:
    print(f"Failed to initialize Ollama client or list models: {e}")
    print("Please ensure Ollama server is running and models are pulled.")
    exit()

Ollama client initialized. Available models include those needed for script if pulled.


Уберем лишние пробелы, переносы строки и тд.:

In [ ]:
text = re.sub(r'-\s* \s*', '', full_text)
text = re.sub(r'\s+', ' ', text).strip()
clean_text = text.strip()
ex = clean_text[5790:7000]
ex

'евича – изобразительная ли сила его таланта или полное незнакомство с вопросом, по которому он собирался писать, – но Иисус в его изображении получился ну совершенно как живой, хотя и не привлекающий к себе персонаж. Берлиоз же хотел доказать поэту, что главное не в том, каков был Иисус, плох ли, хорош ли, а в том, что Иисуса-то этого, как личности, вовсе не существовало на свете и что все рассказы о нем – простые выдумки, самый обыкновенный миф. Надо заметить, что редактор был человеком начитанным и очень умело указывал в своей речи на древних историков, например, на знаменитого Филона Александрийского, на блестяще образованного Иосифа Флавия, никогда ни словом не упоминавших о существовании Иисуса. Обнаруживая солидную эрудицию, Михаил Александрович сообщил поэту, между прочим, и о том, что то место в 15-й книге, в главе 44-й знаменитых Тацитовых «Анналов», где говорится о казни Иисуса, – есть не что иное, как позднейшая поддельная вставка. Поэт, для которого все, сообщаемое редакто

Видим, что текст достаточно чистый, можно продолжать работу дальше

## Быстрая проверка модели

In [ ]:
client = ollama.Client()
system_prompt="Дай ответ на вопрос на русском языке"

def generate_answer_with_gemma(text, system_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"На основе следующего фрагмента текста: '{text}', ответьте на вопрос: что хотел донести Берлиоз"},
    ]


    response = client.chat(
        model="gemma3:4b",
        messages=messages,
        stream=False
    )
    return response.get('message', {}).get('content', "Ответ не найден")

response_text = generate_answer_with_gemma(ex, system_prompt)
print(response_text)


Берлиоз хотел донести, что Иисуса как личности не существовало. Он утверждал, что все рассказы о нем – это просто миф и выдумка, а не исторический факт. Он подкреплял свои слова ссылками на древних историков, таких как Филон Александрийский и Иосиф Флавий, которые никогда не упоминали о существовании Иисуса.



## Split

Для текста хорошо подойдет RecursiveCharacterTextSplitter, так как он сохраняет смысловые блоки максимально целыми и при этом выдерживая заданный размер чанка с нужным overlap

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100,
    separators=["\n\n","\n", ". ", " ", ""]
)
chunks = splitter.split_text(full_text)
len(chunks)

871

## Embed

Выбрала BGE-M3, тк она есть в ollama + ее мы использовали на семинаре. Из плюсов, модель поддерживает русский язык, оптимальна по соотношению размер/качество и контекстное окно эмбеддингов относительно большое, так сказать:
* Multi-Functionality: It can simultaneously perform the three common retrieval functionalities of embedding model: dense retrieval, multi-vector retrieval, and sparse retrieval.
* Multi-Linguality: It can support more than 100 working languages.
* Multi-Granularity: It is able to process inputs of different granularities, spanning from short sentences to long documents of up to 8192 tokens.

In [ ]:
from langchain.embeddings import OllamaEmbeddings
from tqdm import tqdm

embedding_model = OllamaEmbeddings(model="bge-m3")


In [ ]:
embeddings = []
for i in tqdm(range(0, len(chunks), 10), unit = "batch"):
    batch = chunks[i:i+10]
    batch_embeddings = embedding_model.embed_documents(batch)  # cоздаем батч из чанков и векторизуем его
    embeddings.extend(batch_embeddings)


  0%|          | 0/88 [00:00<?, ?batch/s]

100%|██████████| 88/88 [42:16<00:00, 28.82s/batch]


In [ ]:
print(embeddings[:15])

[[0.0819852203130722, 0.6420820951461792, -0.118165522813797, -0.23206551373004913, 0.409184992313385, 0.5152078866958618, 0.7668592929840088, -0.5566183924674988, 0.7400088906288147, 0.21072907745838165, -0.33065202832221985, 1.5306580066680908, -0.20425808429718018, -0.2937013506889343, -0.001403549686074257, -0.0341605618596077, 1.237522006034851, 0.12654587626457214, -0.2951946258544922, -0.7255184054374695, -0.4609052538871765, 0.23500312864780426, -1.1674606800079346, 0.7350389361381531, -0.00869544968008995, 0.6448789834976196, 0.5150895118713379, -0.7327167391777039, -0.39525094628334045, 0.1625945270061493, 0.03297330439090729, -0.836325466632843, -0.7499209046363831, 0.22936949133872986, -0.3907664120197296, -0.7576956152915955, 0.3380604386329651, -0.26402124762535095, -0.8323778510093689, 0.9972991943359375, 0.3888600468635559, -0.34456801414489746, 1.1890326738357544, -2.4760947227478027, -0.4877930283546448, -0.47756892442703247, 0.27466118335723877, -0.8340009450912476, 

## Store & Retrieve

In [ ]:
import chromadb
from langchain_community.vectorstores import Chroma

persist_dir = "chroma_store"
client = chromadb.PersistentClient(path=persist_dir)

collection_name = "scam_calls"
col = client.get_or_create_collection(collection_name)

ids = [f"doc-{i}" for i in range(len(chunks))]
metadatas = [{"idx": i} for i in range(len(chunks))]

# добавляем готовые эмбеддинги
col.add(ids=ids, documents=chunks, embeddings=embeddings, metadatas=metadatas)

# подключаемся через LangChain, чтобы получить retriever
vectorstore = Chroma(
    client=client,
    collection_name=collection_name,
    embedding_function=embedding_model
)
retriever = vectorstore.as_retriever()
print("Готово: коллекция в Chroma заполнена предвычисленными эмбеддингами.")

Готово: коллекция в Chroma заполнена предвычисленными эмбеддингами.


/var/folders/08/x9fwhxhd3bbbxdcvhgpvvh1r0000gn/T/ipykernel_931/4197024412.py:17: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


## Generate & Chain

In [ ]:
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.schema import Document

In [ ]:
SYS_PROMPT_ANSWER_IMPROVED_TEMPLATE = """
Тебе предоставлен КОНТЕКСТ и ВОПРОС.
Твоя задача — дать четкий и развернутый ответ на ВОПРОС, основываясь ИСКЛЮЧИТЕЛЬНО на информации из предоставленного КОНТЕКСТА.
1. Не добавляй никакой информации извне. Не выдумывай детали.
2. Если ответа нет в контексте, скажи: "Я не могу найти ответ в предоставленном тексте."
Ответ должен быть на русском языке.

КОНТЕКСТ:
{context}

ВОПРОС:
{question}

ОТВЕТ:
"""

PROMPT = PromptTemplate(
    template=SYS_PROMPT_ANSWER_IMPROVED_TEMPLATE,
    input_variables=["context", "question"]
)

In [ ]:
llm = Ollama(model="gemma3:4b")

/var/folders/08/x9fwhxhd3bbbxdcvhgpvvh1r0000gn/T/ipykernel_931/1559220570.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="gemma3:4b")


Возьмем несколько «режимов» RAG-цепочек под разные задачи. В частности возьмем 4 основных варианта RetrievalQA:

1) stuff — все найденные чанки «склеиваются» в один промпт и подаются LLM, просто зато быстро

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True
)

2) map_reduce — LLM сначала отвечает по каждому документу отдельно, ака мэппинг, затем агрегирует в финальный ответ (редьюс), лучше агрегирует в сравнении с другими

In [ ]:
MAP_PROMPT = PromptTemplate(
    template="""
Ты ответчик по одному фрагменту из романа.
Отвечай ТОЛЬКО на основе фрагмента.
Если ответа нет — напиши: "Нет ответа в этом фрагменте."

ФРАГМЕНТ:
{context}

ВОПРОС:
{question}

ОТВЕТ:
""",
    input_variables=["context", "question"]
)

REDUCE_PROMPT = PromptTemplate(
    template="""
У тебя есть набор черновых ответов по разным фрагментам.
Синтезируй единый ответ. Если итог всё ещё не подтверждён — скажи:
"Я не могу найти ответ в предоставленном тексте."

ЧЕРНОВЫЕ ОТВЕТЫ:
{summaries}

ВОПРОС:
{question}

ИТОГОВЫЙ ОТВЕТ:
""",
    input_variables=["summaries", "question"]
)

qa_map_reduce = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="map_reduce",
    retriever=retriever,
    chain_type_kwargs={
        "question_prompt": MAP_PROMPT,
        "combine_prompt": REDUCE_PROMPT
    },
    return_source_documents=True
)

3) refine — LLM генерирует первичную версию на первом документе, затем последовательно «уточняет/расширяет» её с каждым следующим документом

In [ ]:
REFINE_INITIAL = PromptTemplate(
    template="""
Сформируй начальный ответ на вопрос ТОЛЬКО по этому фрагменту.
Если нет ответа — скажи: "Нет ответа в этом фрагменте."

ФРАГМЕНТ:
{context}

ВОПРОС:
{question}

НАЧАЛЬНЫЙ ОТВЕТ:
""",
    input_variables=["context", "question"]
)
REFINE_STEP = PromptTemplate(
    template="""
Есть текущий ответ и НОВЫЙ фрагмент. Уточни/дополни ответ ТОЛЬКО при наличии новых фактов.
Если новый фрагмент не помогает — оставь ответ без изменений.
Если ответа всё ещё нет — сохраняй формулировку: "Я не могу найти ответ в предоставленном тексте."

ТЕКУЩИЙ ОТВЕТ:
{existing_answer}

НОВЫЙ ФРАГМЕНТ:
{context}

ВОПРОС:
{question}

УТОЧНЁННЫЙ ОТВЕТ:
""",
    input_variables=["existing_answer", "context", "question"]
)

qa_refine = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="refine",
    retriever=retriever,
    chain_type_kwargs={
        "question_prompt": REFINE_INITIAL,
        "refine_prompt": REFINE_STEP,
        "document_variable_name": "context"
    },
    return_source_documents=True
)


4. LLM отвечает на КАЖДЫЙ фрагмент отдельно и присваивает ему оценку уверенности, должен хорошо справляться с фактологическими вопросами, где нужен факт из конкретного фрагмента

In [ ]:
qa_map_rerank = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="map_rerank",
    retriever=retriever,
    return_source_documents=True
)


In [ ]:

def run(qa, name, question):
    out = qa.invoke({"query": question})
    print(f"\n=== [{name}] ВОПРОС: {question}")
    print("ОТВЕТ:\n", out["result"])
    print("ИСТОЧНИКИ:")
    for i, d in enumerate(out["source_documents"], 1):
        snippet = d.page_content.strip()
        source = (snippet[:200] + "...")
        print(f"({i}): {source}")
        print("-" * 200)

## Пример 1: Прямой поиск факта

In [ ]:
query = "Кто такой Воланд"
run(qa_chain, "stuff", query)


=== [stuff] ВОПРОС: Кто такой Воланд
ОТВЕТ:
 Воланд – это маэстро Воланд, иностранный артист, который предлагает гастроли в Варьете. Он также известен как дух зла и повелитель теней, имя которого носит человек в хитоне, чернобородый, который вышел из круглой башни на крыше. Он также именующий себя Воландом, который бежал за границу, но нигде не появился и ничем себя не проявил.
ИСТОЧНИКИ:
(1): . Дело в том, что в этом вчерашнем дне зияла преогромная черная дыра. Вот этого самого незнакомца в берете, воля ваша, Степа в своем кабинете вчера никак не видал. – Профессор черной магии Воланд, – в...
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
(2): . А он мне отвечает: «А я живу в другой половине!» – Бенгальский сделал паузу, ожидая, что произойдет взрыв смеха, но так как никто не засмеялся, то он продолжал: – … Итак, выступает знамениты

In [ ]:
query = "Кто такой Воланд"

run(qa_map_reduce, "map_reduce", query)


=== [map_reduce] ВОПРОС: Кто такой Воланд
ОТВЕТ:
 Воланд — профессор черной магии, знаменитый иностранный артист, дух зла и повелитель теней. Именующий себя Воландом, он исчез из Москвы и больше не появлялся.

ИСТОЧНИКИ:
(1): . Дело в том, что в этом вчерашнем дне зияла преогромная черная дыра. Вот этого самого незнакомца в берете, воля ваша, Степа в своем кабинете вчера никак не видал. – Профессор черной магии Воланд, – в...
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
(2): . А он мне отвечает: «А я живу в другой половине!» – Бенгальский сделал паузу, ожидая, что произойдет взрыв смеха, но так как никто не засмеялся, то он продолжал: – … Итак, выступает знаменитый иностр...
---------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
query = "Кто такой Воланд"

run(qa_refine, "refine", query)


=== [refine] ВОПРОС: Кто такой Воланд
ОТВЕТ:
 Профессор черной магии, мосье Воланд, дух зла и повелитель теней, именующий себя Воландом, бежал за границу после недавней деятельности.
ИСТОЧНИКИ:
(1): . Дело в том, что в этом вчерашнем дне зияла преогромная черная дыра. Вот этого самого незнакомца в берете, воля ваша, Степа в своем кабинете вчера никак не видал. – Профессор черной магии Воланд, – в...
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
(2): . А он мне отвечает: «А я живу в другой половине!» – Бенгальский сделал паузу, ожидая, что произойдет взрыв смеха, но так как никто не засмеялся, то он продолжал: – … Итак, выступает знаменитый иностр...
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
query = "Кто такой Воланд"

run(qa_map_rerank, "map_rerank", query)

/Users/macs/.pyenv/versions/3.11.12/lib/python3.11/site-packages/langchain/chains/combine_documents/map_rerank.py:192: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  results = self.llm_chain.apply_and_parse(



=== [map_rerank] ВОПРОС: Кто такой Воланд
ОТВЕТ:
 Helpful Answer: Professor of black magic Voland
ИСТОЧНИКИ:
(1): . Дело в том, что в этом вчерашнем дне зияла преогромная черная дыра. Вот этого самого незнакомца в берете, воля ваша, Степа в своем кабинете вчера никак не видал. – Профессор черной магии Воланд, – в...
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
(2): . А он мне отвечает: «А я живу в другой половине!» – Бенгальский сделал паузу, ожидая, что произойдет взрыв смеха, но так как никто не засмеялся, то он продолжал: – … Итак, выступает знаменитый иностр...
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
(3): . Опять наступило молчание, и оба находящихся на террасе глядели,

Так как qa_map_rerank получался хуже с русским промптом, решила оставить английский вариант

И правда Воланд представлялся как профессор черной магии, в принципе все RAG-цепочки дали приемлимый ответ, но наиболее краткий и емкий ответ дала map_rerank, что в принципе ожидаемо, учитывая его специфику, в то время как stuff дал наиболее расплывчатый и нечеткий ответ

## Пример 2: Синтез информации

In [ ]:
chains = [("Stuff", qa_chain),
        ("Map-Reduce", qa_map_reduce),
        ("Refine", qa_refine),
        ("Map-Rerank", qa_map_rerank)]

In [ ]:
query = "Как свита Воланда воздействует на москвичей в романе?"

for name, chain in chains:
    run(chain, name, query)
    print("~" * 200)


=== [Stuff] ВОПРОС: Как свита Воланда воздействует на москвичей в романе?
ОТВЕТ:
 В романе свита Воланда воздействует на москвичей, заставляя их "свиняться", пьянствовать, вступать в связи с женщинами и использовать свое положение. Также свита использует людей, таких как Степан Богданович, и заставляет их совершать поступки, например, "козлиным голосом запевать" о себе в множественном числе. Кроме того, свита вызывает слухи и страх, которые распространяются по всей Москве и провинции.

ИСТОЧНИКИ:
(1): . «Вот как, оказывается, сходят с ума!» – подумал он и ухватился за притолоку. – Я вижу, вы немного удивлены, дражайший Степан Богданович? – осведомился Воланд у лязгающего зубами Степы, – а между тем...
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
(2): . Кто-то отпускал на свободу мастера, как сам он только что отпустил им созданного

/Users/macs/.pyenv/versions/3.11.12/lib/python3.11/site-packages/langchain/chains/combine_documents/map_rerank.py:192: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  results = self.llm_chain.apply_and_parse(



=== [Map-Rerank] ВОПРОС: Как свита Воланда воздействует на москвичей в романе?
ОТВЕТ:
 Helpful Answer: The entourage of Voland is disruptive and chaotic, causing problems such as excessive drinking (the cat drinking vodka), disruptive behavior, and exploiting their positions for personal gain. They are generally unproductive and a nuisance.
ИСТОЧНИКИ:
(1): . «Вот как, оказывается, сходят с ума!» – подумал он и ухватился за притолоку. – Я вижу, вы немного удивлены, дражайший Степан Богданович? – осведомился Воланд у лязгающего зубами Степы, – а между тем...
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
(2): . Кто-то отпускал на свободу мастера, как сам он только что отпустил им созданного героя. Этот герой ушел в бездну, ушел безвозвратно, прощенный в ночь на воскресенье сын короля-звездочета, жестокий п...
--------------------------

In [ ]:
query = "Как свита Воланда воздействует на москвичей в романе?"
run(qa_refine, "refine", query)


=== [refine] ВОПРОС: Как свита Воланда воздействует на москвичей в романе?
ОТВЕТ:
 Свита Воланда воздействует на москвичей посредством создания слухов и запугивания, а также использования людей, в частности Степа Богдановича, и их "жуткого свиняствования" (пьянства и связей). Кроме того, свита вызывает необъяснимые события – исчезновение мостов и дворцов, грозу, и другие стихийные бедствия, что вызывает у людей страх и замешательство.

ИСТОЧНИКИ:
(1): . «Вот как, оказывается, сходят с ума!» – подумал он и ухватился за притолоку. – Я вижу, вы немного удивлены, дражайший Степан Богданович? – осведомился Воланд у лязгающего зубами Степы, – а между тем...
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
(2): . Кто-то отпускал на свободу мастера, как сам он только что отпустил им созданного героя. Этот герой ушел в бездну, ушел безвозвратно

Видим, что map-reduce и map-rerank справились чуть хуже, чем остальные, что опять же ожидаемо в связи со спецификой, зато refine и stuff (особенно stuff) очень хорошо справились с обощением информации по тексту. Лучше всего ответ получился у stuff по моему мнению.


## Пример 3: "Сложный"/"негативный" случай

In [ ]:
query = "Какова настоящая фамилия Мастера?"

for name, chain in chains:
    run(chain, name, query)
    print("~" * 200)

run(qa_refine, "refine", query)


=== [Stuff] ВОПРОС: Какова настоящая фамилия Мастера?
ОТВЕТ:
 Я не могу найти ответ в предоставленном тексте.
ИСТОЧНИКИ:
(1): . – Вы – писатель? – с интересом спросил поэт. Гость потемнел лицом и погрозил Ивану кулаком, потом сказал: – Я – мастер, – он сделался суров и вынул из кармана халата совершенно засаленную черную шап...
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
(2): . Маргарита щурилась на надпись, соображая, что бы могло означать слово «Драмлит». Взяв щетку под мышку, Маргарита вошла в подъезд, толкнув дверью удивленного швейцара, и увидела рядом с лифтом на сте...
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
(3): . Что вы хотите для себя? Наступило молчание, и прерв

/Users/macs/.pyenv/versions/3.11.12/lib/python3.11/site-packages/langchain/chains/combine_documents/map_rerank.py:192: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  results = self.llm_chain.apply_and_parse(



=== [Map-Rerank] ВОПРОС: Какова настоящая фамилия Мастера?
ОТВЕТ:
 The document does not answer the question.
ИСТОЧНИКИ:
(1): . – Вы – писатель? – с интересом спросил поэт. Гость потемнел лицом и погрозил Ивану кулаком, потом сказал: – Я – мастер, – он сделался суров и вынул из кармана халата совершенно засаленную черную шап...
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
(2): . Маргарита щурилась на надпись, соображая, что бы могло означать слово «Драмлит». Взяв щетку под мышку, Маргарита вошла в подъезд, толкнув дверью удивленного швейцара, и увидела рядом с лифтом на сте...
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
(3): . Что вы хотите для себя? Наступило молчание, и прерв

С нетривиальным случаем хорошо справилась только refine цепочка, остальные либо не нашли ответа, либо дали неверный ответ. Refine цепочка же хорошо определила нужный фрагмент, но обобщения какого-то явного не было. Проверим езе на паре вопросов, как работают наши цепочки:

In [ ]:
query = "Почему Воланд забирает Мастера и Маргариту в «покой», а не в «свет»"

for name, chain in chains:
    run(chain, name, query)
    print("~" * 200)

run(qa_refine, "refine", query)


=== [Stuff] ВОПРОС: Почему Воланд забирает Мастера и Маргариту в «покой», а не в «свет»
ОТВЕТ:
 Он не заслужил света, он заслужил покой, – печальным голосом проговорил Левий. – Передай, что будет сделано, – ответил Воланд и прибавил, причем глаз его вспыхнул: – И покинь меня немедленно.
ИСТОЧНИКИ:
(1): . – Зачем? – продолжал Воланд убедительно и мягко, – о, трижды романтический мастер, неужто вы не хотите днем гулять со своею подругой под вишнями, которые начинают зацветать, а вечером слушать музыку...
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
(2): . Итак… – Он прочитал сочинение мастера, – заговорил Левий Матвей, – и просит тебя, чтобы ты взял с собою мастера и наградил его покоем. Неужели это трудно тебе сделать, дух зла? – Мне ничего не трудн...
---------------------------------------------------------------------------------

/Users/macs/.pyenv/versions/3.11.12/lib/python3.11/site-packages/langchain/chains/combine_documents/map_rerank.py:192: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  results = self.llm_chain.apply_and_parse(



=== [Map-Rerank] ВОПРОС: Почему Воланд забирает Мастера и Маргариту в «покой», а не в «свет»
ОТВЕТ:
 Воланд забирает Мастера и Маргариту в "покой", а не в "свет", потому что Левий Матвей считает, что Мастер заслужил покой, а не свет. Воланд соглашается с этим, поскольку Мастер не заслужил света, а скорее заслужит покой.
ИСТОЧНИКИ:
(1): . – Зачем? – продолжал Воланд убедительно и мягко, – о, трижды романтический мастер, неужто вы не хотите днем гулять со своею подругой под вишнями, которые начинают зацветать, а вечером слушать музыку...
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
(2): . Итак… – Он прочитал сочинение мастера, – заговорил Левий Матвей, – и просит тебя, чтобы ты взял с собою мастера и наградил его покоем. Неужели это трудно тебе сделать, дух зла? – Мне ничего не трудн...
-----------------------------------------------

В принципе, так или иначе справились все, stuff и map_rerank по сути воспроизвели найденный фрагмент, map_reduce как и ожидается собрал ответ из нескольких кусочков (тк упоминается и Воланд и «ещё лучше»), но потерял в ответе Левия, refine тоже выдал связный ответ, но добавил свою интерпретацию, которой нет в романе => может быть риск галлюцинации